# PA2 — Inferência

Recebe o caminho de **uma sequência qualquer** e devolve o **vídeo com as
identidades coloridas de forma consistente** e a **contagem de objetos únicos**.

Não treina nada: carrega um checkpoint já treinado do modelo temporal.

## 1. Ambiente

No Colab, descomente o bloco do clone. Localmente, basta rodar a partir de `PA2/`.

In [ ]:
# --- Colab ---
# !git clone https://github.com/sofiaazeredo/deep-learning-2026.2.git
# %cd deep-learning-2026.2/PA2

import sys
from pathlib import Path

if Path("src").exists() and "." not in sys.path:
    sys.path.insert(0, ".")

import numpy as np
import matplotlib.pyplot as plt

from src.inference import track_sequence, draw_tracks, write_video

## 2. Escolha a sequência

Troque `SEQUENCE_PATH` por qualquer pasta no layout do MOT17
(`img1/`, e `det/det.txt` se for usar as detecções públicas).

`CHECKPOINT = None` usa o modelo final da ablação
(`experiments/results/best_model.json`); passe um caminho para forçar outro.

In [ ]:
SEQUENCE_PATH = Path("data/MOT17/train/MOT17-02-SDP")

CHECKPOINT = None     # None = o modelo final da ablação
DETECTOR = "SDP"      # fonte de detecções: "DPM", "FRCNN", "SDP" ou "torchvision"
WINDOW = None         # None processa a sequência inteira; um inteiro roda em janelas

print("sequência:", SEQUENCE_PATH)

## 3. Rastreamento

In [ ]:
result = track_sequence(
    SEQUENCE_PATH,
    checkpoint=CHECKPOINT,
    detector=DETECTOR,
)

print("objetos únicos no vídeo:", result["count"])

## 4. Quadros com as identidades coloridas

Uma cor estável por identidade: se a mesma cor reaparece depois de uma oclusão,
a identidade sobreviveu.

In [ ]:
frames = result["frames"]
indices = np.linspace(0, len(frames) - 1, 6).astype(int)

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, i in zip(axes.ravel(), indices):
    ax.imshow(frames[i])
    ax.set_title(f"quadro {i}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Vídeo

In [ ]:
OUTPUT = Path("experiments/figures") / f"{SEQUENCE_PATH.name}_tracks.mp4"

write_video(frames, OUTPUT, fps=30)
print("vídeo salvo em:", OUTPUT)

## 6. Trajetórias previstas

Formato `frame, id, x, y, w, h` — o mesmo que `src/metrics.py` consome, então
dá para avaliar esta saída direto com `scripts/evaluate_tracking.py`.

In [ ]:
tracks = result["tracks"]
print(f"{len(tracks)} caixas previstas em {len(frames)} quadros")
tracks[:10]